In [1]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LogisticRegressionCV as LogRegcv
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_table('sim_data.txt', sep=' ')
df

,V1,V2,V3,V4,V5,covar
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,1,1,0,0,0,1
4,1,0,0,0,1,1
5,0,0,0,1,1,1
...,...,...,...,...,...,...
4996,0,0,0,0,1,0
4997,0,1,0,0,0,0
4998,1,0,0,1,0,1
4999,1,1,0,0,1,1


In [3]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [4]:
model = LogRegcv(cv=10, max_iter=1000, penalty='l1', solver='saga',n_jobs=-1, Cs=70)

In [5]:
J_0 = []
h_0 = []
J_y = []
h_y = []

for idx, col in enumerate(X.columns):
    print(f'Fitting variable {idx+1} out of {X.shape[1]}')
    
    target = np.array(X.iloc[:, idx])
    features = np.array(X.iloc[:, [i for i in range(X.shape[1]) if i != idx]])
    covar = np.array(y).reshape(-1,1)
    interactions = features * covar
    X_model = np.hstack((features, covar, interactions))
    
    fit_ = model.fit(X_model, target)
    
    print(fit_.intercept_)
    print(fit_.coef_)
    
    h_0.append(fit_.intercept_[0])
    j0 = fit_.coef_[0, :features.shape[1]]
    j0 = np.insert(j0, idx, 0)
    J_0.append(j0)
    
    h_y.append(fit_.coef_[0, features.shape[1]])
    jy = fit_.coef_[0, features.shape[1]+1:]
    jy = np.insert(jy, idx, 0)
    J_y.append(jy)
    
J_0 = np.vstack(J_0)
h_0 = np.vstack(h_0)
J_y = np.vstack(J_y)
h_y = np.vstack(h_y)

Fitting variable 1 out of 5
[-0.63607661]
[[-1.419165   -1.31605293 -1.72487288 -1.80269127  4.10631121  0.
   0.          0.          0.04944791]]
Fitting variable 2 out of 5
[-1.45660223]
[[-1.79809411 -1.55066709 -1.7032177  -2.03400153  4.55501287  0.
   0.          0.          0.        ]]
Fitting variable 3 out of 5
[-2.4220126]
[[-2.0526007  -1.84213869 -2.00503801 -2.15118583  4.86023501  0.
   0.          0.          0.        ]]
Fitting variable 4 out of 5
[-1.06349335]
[[-4.07985302 -3.22345803 -2.14849543 -4.39187181  4.96928415  1.82361693
   1.37896117  0.13574123  2.37443443]]
Fitting variable 5 out of 5
[-0.22955044]
[[0.         0.         0.         0.         0.45098224 0.
  0.         0.         0.        ]]


In [6]:
J_0 = []
h_0 = []
J_y = []
h_y = []

X = X.values
y = y.values.reshape(-1,1)

nvars = X.shape[1]

for idx in range(nvars):
    print(f'Fitting variable {idx+1} out of {nvars}')
    
    target = X[:, idx]
    features = np.delete(X, idx, axis=1)
    covar = y
    interactions = features * covar
    X_model = np.hstack((features, covar, interactions))
    
    fit_ = model.fit(X_model, target)
    
    print(fit_.intercept_)
    print(fit_.coef_)
    
    h_0.append(fit_.intercept_[0])
    j0 = fit_.coef_[0, :features.shape[1]]
    j0 = np.insert(j0, idx, 0)
    J_0.append(j0)
    
    h_y.append(fit_.coef_[0, features.shape[1]])
    jy = fit_.coef_[0, features.shape[1]+1:]
    jy = np.insert(jy, idx, 0)
    J_y.append(jy)
    
J_0 = np.vstack(J_0)
h_0 = np.vstack(h_0)
J_y = np.vstack(J_y)
h_y = np.vstack(h_y)

Fitting variable 1 out of 5
[-0.63585996]
[[-1.41925516 -1.31593173 -1.72475486 -1.80259411  4.10621406  0.
   0.          0.          0.04957289]]
Fitting variable 2 out of 5
[-1.4563209]
[[-1.79817807 -1.55108927 -1.70383618 -2.03395498  4.55524707  0.
   0.          0.          0.        ]]
Fitting variable 3 out of 5
[-2.42206019]
[[-2.05255673 -1.84221419 -2.00495231 -2.15125394  4.86018213  0.
   0.          0.          0.        ]]
Fitting variable 4 out of 5
[-1.06347229]
[[-4.08005054 -3.22418309 -2.14803895 -4.39245283  4.96923189  1.82378192
   1.37968245  0.13530165  2.37502064]]
Fitting variable 5 out of 5
[-0.23038242]
[[0.         0.         0.         0.         0.45044817 0.
  0.         0.         0.        ]]


In [16]:
print(X, "\n", h_0, "\n", X @ h_0)

[[0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]
 ...
 [1 0 0 1 0]
 [1 1 0 0 1]
 [0 0 0 0 1]] 
 [[-0.63585996]
 [-1.4563209 ]
 [-2.42206019]
 [-1.06347229]
 [-0.23038242]] 
 [[ 0.        ]
 [ 0.        ]
 [-2.09218085]
 ...
 [-1.69933224]
 [-2.32256327]
 [-0.23038242]]


In [29]:
np.einsum('ij,jk-> i', X, h_0)

array([ 0.        ,  0.        , -2.09218085, ..., -1.69933224,
       -2.32256327, -0.23038242], shape=(5000,))

In [23]:
print(X, "\n", y.reshape(-1, 1), "\n", h_y, "\n", X * y.reshape(-1, 1) @ h_y)

[[0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]
 ...
 [1 0 0 1 0]
 [1 1 0 0 1]
 [0 0 0 0 1]] 
 [[0]
 [0]
 [1]
 ...
 [1]
 [1]
 [0]] 
 [[4.10621406]
 [4.55524707]
 [4.86018213]
 [4.96923189]
 [0.45044817]] 
 [[0.        ]
 [0.        ]
 [8.66146112]
 ...
 [9.07544595]
 [9.11190929]
 [0.        ]]


In [35]:
np.einsum('ij,jk-> i', X, h_0) + np.einsum('ij,jk-> i', X * y.reshape(-1, 1), h_y)

array([ 0.        ,  0.        ,  6.56928027, ...,  7.37611371,
        6.78934602, -0.23038242], shape=(5000,))

In [36]:
np.einsum('ij,jk,ik-> i', X, J_0, X) * 0.5

array([ 0.        ,  0.        , -1.60871662, ..., -2.9024027 ,
       -3.52699116,  0.        ], shape=(5000,))

In [37]:
np.einsum('ij,jk,ik-> i', X * y.reshape(-1, 1), J_y, X * y.reshape(-1, 1)) * 0.5

array([0.        , 0.        , 0.        , ..., 0.91189096, 0.02478644,
       0.        ], shape=(5000,))

In [31]:
X[0:5,:] @ J_0 @ X[0:5,:].T

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -3.21743324, -5.63472716, -7.26514013],
       [ 0.        ,  0.        , -1.41925516, -1.80259411, -3.52734897],
       [ 0.        ,  0.        , -7.30423364, -8.47250337, -4.39245283]])

In [41]:
print(X.values[0:10], "\n", y.values.reshape(-1,1)[0:10], "\n", X.values[0:10] * y.values.reshape(-1,1)[0:10])

[[0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]
 [1 0 0 0 1]
 [0 0 0 1 1]
 [0 1 0 0 1]
 [0 0 0 0 0]
 [0 1 0 1 1]
 [0 0 0 0 0]
 [1 0 0 0 0]] 
 [[0]
 [0]
 [1]
 [1]
 [1]
 [1]
 [0]
 [1]
 [0]
 [0]] 
 [[0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]
 [1 0 0 0 1]
 [0 0 0 1 1]
 [0 1 0 0 1]
 [0 0 0 0 0]
 [0 1 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]


In [6]:
print(h_0,"\n",J_0,"\n", J_y,"\n", h_y)

[[-0.63593012]
 [-1.45650645]
 [-2.42210366]
 [-1.06347448]
 [-0.22955948]] 
 [[ 0.         -1.41899481 -1.31587299 -1.7248461  -1.80273308]
 [-1.79825331  0.         -1.55069798 -1.70381005 -2.03396377]
 [-2.05242999 -1.84218485  0.         -2.00498044 -2.15129343]
 [-4.08005894 -3.22365283 -2.14847689  0.         -4.39207635]
 [ 0.          0.          0.          0.          0.        ]] 
 [[0.         0.         0.         0.         0.04954337]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [1.82378599 1.37914286 0.1357463  0.         2.37462082]
 [0.         0.         0.         0.         0.        ]] 
 [[4.10614393]
 [4.55523715]
 [4.860131  ]
 [4.96928672]
 [0.45099847]]


In [7]:
# Criterion: max
print(np.where(np.abs(J_0) >= np.abs(J_0.T), J_0, J_0.T))


[[ 0.         -1.79825331 -2.05242999 -4.08005894 -1.80273308]
 [-1.79825331  0.         -1.84218485 -3.22365283 -2.03396377]
 [-2.05242999 -1.84218485  0.         -2.14847689 -2.15129343]
 [-4.08005894 -3.22365283 -2.14847689  0.         -4.39207635]
 [-1.80273308 -2.03396377 -2.15129343 -4.39207635  0.        ]]


In [9]:
# Criterion: min
print(np.where(np.abs(J_0) <= np.abs(J_0.T), J_0, J_0.T))

[[ 0.         -1.41899481 -1.31587299 -1.7248461   0.        ]
 [-1.41899481  0.         -1.55069798 -1.70381005  0.        ]
 [-1.31587299 -1.55069798  0.         -2.00498044  0.        ]
 [-1.7248461  -1.70381005 -2.00498044  0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]]


In [24]:
s0 = np.array(X.iloc[0:5, :])
sy = np.array(y.iloc[0:5])

In [ ]:
# interaction_effects 
np.einsum('ij, jk, ik -> i', s0, J_0, s0)

In [ ]:
# main effects
np.einsum('ij, jk -> i', s0, h_0)

In [26]:
# covar effect 
np.einsum('ij, jk -> i', sy, h_y)

ValueError: einstein sum subscripts string contains too many subscripts for operand 0